# Library & Knowledge Resource Access Control System
**CT-351 — Knowledge Representation & Reasoning | Complex Computing Problem (CCP)**

**Google Colab executable project.** This notebook implements the same nine Horn-clause rules and knowledge base as the accompanying `library_kr.py` file, with forward chaining, backward chaining, and sample queries.

**Start:** Click **Runtime → Run all** in Google Colab. You do **not** need to upload files or install Python libraries. The data are fictional, for classroom demonstration only; this is not connected to the NED University library.

**Group 4:** Anam Mansoor (AI-23304), Duryab Anwar (AI-23305), Hareem Aqeel (AI-23006), Atika Ali (AI-23013).

## 1. Knowledge base and inference engine

- **Facts** describe patrons, resources, and permissions-related attributes.
- **Horn clauses (R1–R9)** encode access policies.
- **Forward chaining** repeatedly derives conclusions from known facts.
- **Backward chaining** checks whether a requested permission can be proved.
- **Default-deny policy:** if permission cannot be proved from available facts, access is not granted. It does not claim that the opposite fact has been established.

Run the following cell first.

In [ ]:
"""CT-351 CCP - Library & Knowledge Resource Access Control System.

A small, transparent Horn-clause engine (standard Python only). The snapshot
contains illustrative facts, not actual university or library records.

Run:
    python library_kr.py --demo
    python library_kr.py sara book_ai borrow --trace
    python library_kr.py omar journal_faculty digital --trace
    python library_kr.py nida book_ai borrow --trace
"""

from __future__ import annotations

import argparse
from dataclasses import dataclass
from typing import Dict, Iterator, List, Optional, Sequence, Set, Tuple

Atom = Tuple[str, ...]
Bindings = Dict[str, str]


@dataclass(frozen=True)
class Rule:
    name: str
    head: Atom
    body: Tuple[Atom, ...]


def a(predicate: str, *args: str) -> Atom:
    return (predicate, *args)


# The listed statements are explicitly present in the test scenario. No
# membership dates, fine balances or external databases are inferred.
BASE_FACTS: Set[Atom] = {
    a("member", "sara"), a("member", "omar"),
    a("member", "nida"), a("member", "hamza"),
    a("student", "sara"), a("faculty", "omar"),
    a("student", "nida"), a("student", "hamza"),
    a("membership_active", "sara"), a("membership_active", "omar"),
    a("membership_active", "hamza"),
    a("account_clear", "sara"), a("account_clear", "omar"),
    a("account_clear", "nida"), a("account_clear", "hamza"),
    a("under_loan_limit", "sara"), a("under_loan_limit", "omar"),
    a("in_library", "sara"), a("in_library", "omar"),
    a("in_library", "hamza"),
    a("authenticated", "sara"), a("authenticated", "omar"),
    a("authenticated", "nida"), a("authenticated", "hamza"),
    a("physical", "book_ai"), a("physical", "book_os"),
    a("physical", "atlas"),
    a("general_loan", "book_ai"), a("general_loan", "book_os"),
    a("copy_available", "book_ai"),
    a("reference_only", "atlas"),
    a("digital", "ebook_ml"), a("digital", "journal_faculty"),
    a("digital", "journal_expired"),
    a("student_license", "ebook_ml"),
    a("faculty_license", "journal_faculty"),
    a("faculty_license", "journal_expired"),
    a("license_active", "ebook_ml"),
    a("license_active", "journal_faculty"),
}

RULES: Tuple[Rule, ...] = (
    Rule("R1", a("active_patron", "?p"),
         (a("member", "?p"), a("membership_active", "?p"))),
    Rule("R2", a("entitled", "?p", "?r"),
         (a("student", "?p"), a("student_license", "?r"))),
    Rule("R3", a("entitled", "?p", "?r"),
         (a("faculty", "?p"), a("faculty_license", "?r"))),
    Rule("R4", a("borrow_allowed", "?p", "?r"),
         (a("active_patron", "?p"), a("account_clear", "?p"),
          a("under_loan_limit", "?p"), a("physical", "?r"),
          a("general_loan", "?r"), a("copy_available", "?r"))),
    Rule("R5", a("onsite_allowed", "?p", "?r"),
         (a("active_patron", "?p"), a("in_library", "?p"),
          a("physical", "?r"), a("reference_only", "?r"))),
    Rule("R6", a("digital_allowed", "?p", "?r"),
         (a("active_patron", "?p"), a("authenticated", "?p"),
          a("digital", "?r"), a("license_active", "?r"),
          a("entitled", "?p", "?r"))),
    Rule("R7", a("can_access", "?p", "?r"),
         (a("borrow_allowed", "?p", "?r"),)),
    Rule("R8", a("can_access", "?p", "?r"),
         (a("onsite_allowed", "?p", "?r"),)),
    Rule("R9", a("can_access", "?p", "?r"),
         (a("digital_allowed", "?p", "?r"),)),
)

ACTIONS = {"borrow": "borrow_allowed", "onsite": "onsite_allowed",
           "digital": "digital_allowed"}
PATRONS = ("sara", "omar", "nida", "hamza")
RESOURCES = ("book_ai", "book_os", "atlas", "ebook_ml",
             "journal_faculty", "journal_expired")


def is_variable(term: str) -> bool:
    return term.startswith("?")


def resolve(term: str, env: Bindings) -> str:
    while is_variable(term) and term in env and env[term] != term:
        term = env[term]
    return term


def unify(pattern: Atom, ground: Atom, bindings: Bindings) -> Optional[Bindings]:
    """Match a rule atom with a ground fact or a partially bound goal."""
    if len(pattern) != len(ground) or pattern[0] != ground[0]:
        return None
    result = dict(bindings)
    for left, right in zip(pattern[1:], ground[1:]):
        left = resolve(left, result)
        right = resolve(right, result)
        if left == right:
            continue
        if is_variable(left):
            result[left] = right
        elif is_variable(right):
            result[right] = left
        else:
            return None
    return result


def instantiate(atom: Atom, env: Bindings) -> Atom:
    return (atom[0], *(resolve(term, env) for term in atom[1:]))


def match_body(body: Sequence[Atom], facts: Set[Atom],
               bindings: Bindings, index: int = 0,
               support: Tuple[Atom, ...] = ()) -> Iterator[Tuple[Bindings, Tuple[Atom, ...]]]:
    if index == len(body):
        yield bindings, support
        return
    for fact in sorted(facts):
        match = unify(body[index], fact, bindings)
        if match is not None:
            yield from match_body(body, facts, match, index + 1, support + (fact,))


def forward_chain(initial_facts: Set[Atom] = BASE_FACTS):
    """Compute the least fixed point; save the first proof of every new fact."""
    facts = set(initial_facts)
    provenance: Dict[Atom, Tuple[str, Tuple[Atom, ...]]] = {}
    trace: List[Tuple[int, str, Atom, Tuple[Atom, ...]]] = []
    round_no = 0
    while True:
        round_no += 1
        additions = {}
        for rule in RULES:
            for env, support in match_body(rule.body, facts, {}):
                conclusion = instantiate(rule.head, env)
                if any(is_variable(term) for term in conclusion[1:]):
                    raise ValueError(f"Unsafe rule {rule.name}: {conclusion}")
                if conclusion not in facts and conclusion not in additions:
                    additions[conclusion] = (rule.name, support)
        if not additions:
            break
        for fact, (name, support) in sorted(additions.items()):
            facts.add(fact)
            provenance[fact] = (name, support)
            trace.append((round_no, name, fact, support))
    return facts, provenance, trace


def backward_proof(goal: Atom, initial_facts: Set[Atom] = BASE_FACTS,
                   seen: Optional[Set[Atom]] = None):
    """Goal-driven depth-first proof, returning a nested proof tree or None.

    All example queries are ground and the rule dependency graph is acyclic.
    """
    seen = set() if seen is None else set(seen)
    if goal in initial_facts:
        return ("FACT", goal, ())
    if goal in seen:
        return None
    seen.add(goal)
    for rule in RULES:
        binding = unify(rule.head, goal, {})
        if binding is None:
            continue
        children = []
        for premise in rule.body:
            concrete = instantiate(premise, binding)
            if any(is_variable(t) for t in concrete[1:]):
                # This example's queries bind all variables from the rule head.
                children = []
                break
            proof = backward_proof(concrete, initial_facts, seen)
            if proof is None:
                children = []
                break
            children.append(proof)
        if len(children) == len(rule.body):
            return (rule.name, goal, tuple(children))
    return None


def fact_str(fact: Atom) -> str:
    return f"{fact[0]}({', '.join(fact[1:])})"


def print_tree(tree, indent=0):
    rule, atom, children = tree
    print("  " * indent + f"{fact_str(atom)}  [{rule}]")
    for child in children:
        print_tree(child, indent + 1)


def explain_failure(goal: Atom, known: Set[Atom]) -> List[str]:
    """Show unmet premises for each matching policy, not invented negative facts."""
    explanations = []
    for rule in RULES:
        env = unify(rule.head, goal, {})
        if env is None:
            continue
        missing = [fact_str(instantiate(req, env)) for req in rule.body
                   if instantiate(req, env) not in known]
        explanations.append(f"{rule.name}: missing " + ", ".join(missing))
    return explanations


def run_query(patron: str, resource: str, action: str, trace: bool = False) -> bool:
    goal = a(ACTIONS[action], patron, resource)
    known, provenance, steps = forward_chain()
    forward_result = goal in known
    proof = backward_proof(goal)
    assert forward_result == (proof is not None), "Chaining engines disagree"
    print(f"QUERY: {fact_str(goal)}")
    print(f"DECISION: {'ALLOWED' if forward_result else 'NOT PROVEN (deny by default)'}")
    print(f"FORWARD CHECK: {'derived' if forward_result else 'not derived'}")
    print(f"BACKWARD CHECK: {'proof found' if proof else 'no proof'}")
    if trace:
        if proof:
            print("BACKWARD PROOF TREE:")
            print_tree(proof)
            print("FORWARD DERIVATIONS USED:")
            def show(atom: Atom, visited: Set[Atom]):
                if atom in visited or atom not in provenance:
                    return
                visited.add(atom)
                name, supporting = provenance[atom]
                for premise in supporting:
                    show(premise, visited)
                print(f"  {name}: {fact_str(atom)} <- " +
                      ", ".join(map(fact_str, supporting)))
            show(goal, set())
        else:
            print("UNSATISFIED POLICY PREMISES:")
            for item in explain_failure(goal, known):
                print("  " + item)
    return forward_result


def run_demo():
    cases = [
        ("sara", "book_ai", "borrow"),
        ("sara", "book_os", "borrow"),
        ("sara", "atlas", "onsite"),
        ("sara", "ebook_ml", "digital"),
        ("sara", "journal_faculty", "digital"),
        ("omar", "journal_faculty", "digital"),
        ("omar", "journal_expired", "digital"),
        ("nida", "book_ai", "borrow"),
        ("hamza", "book_ai", "borrow"),
        ("hamza", "atlas", "onsite"),
    ]
    facts, _, steps = forward_chain()
    print("ILLUSTRATIVE LIBRARY ACCESS CASES")
    print("-" * 74)
    for patron, resource, action in cases:
        goal = a(ACTIONS[action], patron, resource)
        assert (goal in facts) == (backward_proof(goal) is not None)
        print(f"{patron:7} | {resource:16} | {action:7} | " +
              ("ALLOW" if goal in facts else "DENY"))
    print("-" * 74)
    print(f"Base facts: {len(BASE_FACTS)} | rules: {len(RULES)} | "
          f"derived facts: {len(steps)} | fixed-point rounds: {max((s[0] for s in steps), default=0)}")
    print("DENY means permission was not proven from the present snapshot.")


def main():
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument("patron", nargs="?", choices=PATRONS)
    parser.add_argument("resource", nargs="?", choices=RESOURCES)
    parser.add_argument("action", nargs="?", choices=tuple(ACTIONS))
    parser.add_argument("--trace", action="store_true", help="print inference details")
    parser.add_argument("--demo", action="store_true", help="run ten sample cases")
    args = parser.parse_args()
    if args.demo or not (args.patron or args.resource or args.action):
        run_demo()
        return
    if not all((args.patron, args.resource, args.action)):
        parser.error("patron, resource and action must all be supplied")
    run_query(args.patron, args.resource, args.action, args.trace)

## 2. Demonstration: access-control decisions

Shows ten illustrative cases for borrowing, reference-only material, and digital licenses. **ALLOW** means the policy was proved; **DENY** means it could not be proved from this knowledge base.

In [ ]:
run_demo()

## 3. Forward chaining: step-by-step trace

This prints the first several new conclusions. Each is justified by a rule and its supporting facts.

In [ ]:
known, provenance, steps = forward_chain()
print(f"Initial facts: {len(BASE_FACTS)} | Rule count: {len(RULES)} | Newly derived facts: {len(steps)}")
for round_num, rule, fact, support in steps[:18]:
    premises = " ∧ ".join(fact_str(item) for item in support)
    print(f"Round {round_num} | {rule}: {premises} → {fact_str(fact)}")

## 4. Backward and forward chaining: an allowed request

Observe how a specific decision follows from the facts. The proof tree lists the exact rules used.

In [ ]:
run_query("sara", "book_ai", "borrow", trace=True)

## 5. Denied request: missing required facts

A different request illustrates why a permission cannot be proved. Missing premises are not silently treated as false facts.

In [ ]:
run_query("nida", "book_ai", "borrow", trace=True)

## 6. Try a custom query

Edit the three strings and rerun this cell.

**Patrons:** `sara`, `omar`, `nida`, `hamza`  
**Resources:** `book_ai`, `book_os`, `atlas`, `ebook_ml`, `journal_faculty`, `journal_expired`  
**Actions:** `borrow`, `onsite`, `digital`

In [ ]:
patron = "omar"                 # change to a patron ID
resource = "journal_faculty"   # change to a resource ID
action = "digital"             # borrow / onsite / digital

assert patron in PATRONS, "Invalid patron ID"
assert resource in RESOURCES, "Invalid resource ID"
assert action in ACTIONS, "Invalid action"
run_query(patron, resource, action, trace=True)

## 7. Automated validation — 13 unit tests

The tests compare the output of forward and backward chaining for allowed and denied cases, including expired digital licenses, unavailable books, inactive membership, and loan limits. They are based on the same tests as `tests/test_library_kr.py`.

In [ ]:
"""Unit tests for the CT-351 library reasoning prototype."""
import unittest


class LibraryPoliciesTest(unittest.TestCase):
    @classmethod
    def setUpClass(cls):
        cls.closure, _, cls.steps = forward_chain()

    def check(self, patron, resource, permission, expected):
        query = a(permission, patron, resource)
        self.assertEqual(query in self.closure, expected)
        self.assertEqual(backward_proof(query) is not None, expected)

    def test_borrow_allow(self): self.check("sara", "book_ai", "borrow_allowed", True)
    def test_unavailable_copy(self): self.check("sara", "book_os", "borrow_allowed", False)
    def test_onsite_reference(self): self.check("sara", "atlas", "onsite_allowed", True)
    def test_student_ebook(self): self.check("sara", "ebook_ml", "digital_allowed", True)
    def test_student_not_faculty_licensed(self): self.check("sara", "journal_faculty", "digital_allowed", False)
    def test_faculty_journal(self): self.check("omar", "journal_faculty", "digital_allowed", True)
    def test_expired_license(self): self.check("omar", "journal_expired", "digital_allowed", False)
    def test_inactive_membership(self): self.check("nida", "book_ai", "borrow_allowed", False)
    def test_loan_limit(self): self.check("hamza", "book_ai", "borrow_allowed", False)
    def test_onsite_for_loan_limited(self): self.check("hamza", "atlas", "onsite_allowed", True)
    def test_reference_not_borrowable(self): self.check("sara", "atlas", "borrow_allowed", False)
    def test_no_reference_online(self): self.check("sara", "atlas", "digital_allowed", False)

    def test_no_unsafe_rules(self):
        self.assertGreater(len(RULES), 0)
        self.assertGreater(len(self.steps), 0)
        self.assertTrue(BASE_FACTS.issubset(self.closure))

suite = unittest.defaultTestLoader.loadTestsFromTestCase(LibraryPoliciesTest)
result = unittest.TextTestRunner(verbosity=2).run(suite)
assert result.wasSuccessful(), "One or more unit tests failed"

## 8. Project files and scope

The downloadable ZIP includes the original **`library_kr.py`**, **`tests/test_library_kr.py`**, architecture/semantic-network diagrams, sample trace files, this Colab notebook, README, and the final report.

**Important:** The system is an educational decision-support prototype using illustrative data. It does not connect to a real catalogue, authentication service, or library database, and it does not actually issue loans.